In [ ]:
import pandas as pd
import psycopg2
import pandas as pd
import geopandas as gpd
import requests
from geopandas.tools import sjoin
import time
import datetime
from collections import Counter 
import re
import math


In [ ]:
def calculer_distance_haversine(lat1, lon1, lat2, lon2):
    # Rayon de la Terre en kilomètres
    R = 6371.0

    # Conversion des degrés en radians
    dLat = math.radians(lat2 - lat1)
    dLon = math.radians(lon2 - lon1)
    rLat1 = math.radians(lat1)
    rLat2 = math.radians(lat2)

    # Formule de Haversine
    a = math.sin(dLat / 2)**2 + math.cos(rLat1) * math.cos(rLat2) * math.sin(dLon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    # Distance totale
    distance = R * c
    return distance

In [ ]:
df_bruit_no_num = pd.read_csv("./gendarmerie/gendarmerie_geocoded_bruite_no_num.csv", sep=";")
df_bruite_50_geocoded = pd.read_csv("./gendarmerie/gendarmerie_geocoded_bruite_50.csv", sep=";")
df = pd.read_csv("./gendarmerie/gendarmerie_geocoded.csv", sep=";")

In [ ]:
df = df_bruit_no_num

for i in df.index :
    lon1 = df.at[i,"geocodage_x_GPS"]
    lat1 = df.at[i,"geocodage_y_GPS"]

    lon2 = df.at[i,"x"]
    lat2 = df.at[i,"y"]

    df.at[i, "distance_m"] = calculer_distance_haversine(lat1, lon1, lat2, lon2)


df_no_num = df[["identifiant_public_unite", "score", "distance_m"]]

In [ ]:
df = df_bruite_50_geocoded

for i in df.index :
    lon1 = df.at[i,"geocodage_x_GPS"]
    lat1 = df.at[i,"geocodage_y_GPS"]

    lon2 = df.at[i,"x"]
    lat2 = df.at[i,"y"]

    df.at[i, "distance_m"] = calculer_distance_haversine(lat1, lon1, lat2, lon2)


df_bruite = df[["identifiant_public_unite", "score", "distance_m"]]

In [ ]:
for i in df.index :
    lon1 = df.at[i,"geocodage_x_GPS"]
    lat1 = df.at[i,"geocodage_y_GPS"]

    lon2 = df.at[i,"x"]
    lat2 = df.at[i,"y"]

    df.at[i, "distance_m"] = calculer_distance_haversine(lat1, lon1, lat2, lon2)


df = df[["identifiant_public_unite", "score", "distance_m"]]

## Impact du score par rapport à la distance 

##### Question : évaluer le coefficient de corrélation entre les deux variables ?

In [ ]:
import plotly.express as px 

In [ ]:
fig = px.scatter(df, x="distance_m", y="score", title = "Impact du score par rapport à la distance (en mètres) pour les données géocodées sans numéros")
fig.show()

In [ ]:
fig = px.scatter(df_no_num, x="distance_m", y="score", title = "Impact du score par rapport à la distance (en mètres) pour les données géocodées sans numéros")
fig.show()

In [ ]:
fig = px.scatter(df_bruite, x="score", y="distance", title = "Impact du score par rapport à la distance (en mètres) pour les données géocodées avec du bruits")
fig.show()

### Calcul du coefficient de corrélation entre le score et la distance 

In [ ]:
coef_correlation_bruit = df_bruite['score'].corr(df_bruite['distance_m'], method='pearson')